In [1]:
from dotenv import load_dotenv
load_dotenv()

import os

if os.environ['GROQ_API_KEY']:
    print("API Key is Set!!!")
else:
    raise ValueError("API Key not loaded!!!")

API Key is Set!!!


In [2]:
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F12F6B6420>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F130906330>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### **Tools**

### DuckDuckGo Search

In [17]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

@tool
def duckduckgo_tool(query:str):
    """This tool searches the duckduckgo for query"""

    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)

In [26]:
duckduckgo_tool.invoke("Anirudh Ravichander")

'Anirudh Ravichander is an Indian composer and playback singer who works in Tamil, Telugu and Hindi films. He is known for his viral songs like "Why This Kolaveri Di" and "Selfie Pulla", and has won several awards and collaborated with various artists. A comprehensive list of films, songs and albums composed and produced by Anirudh Ravichander, a popular Indian music director. Find out his debut, upcoming projects, collaborations and awards in Tamil, Telugu and Hindi languages. Anirudh Ravichander (born 16 October 1990) is an Indian composer, playback singer, instrumentalist, and music producer who primarily works in Tamil cinema, with contributions to... 12M Followers, 492 Following, 1,921 Posts - Anirudh (@AnirudhOfficial) on Instagram: "@recordsalbuquerque @pieceofrockoffl @localokatequila" Oct 16, 2024 · Anirudh is the son of Tamil actor Ravi Raghavendra. He is the nephew of superstar Rajinikanth. He started composing music at the age of 10. He made his debut at the age of 21 in th

### ArXiv Tool

In [27]:
import arxiv

@tool
def arxiv_tool(query: str) -> str:
    """
    Search arXiv and return summaries of the top papers.
    """

    client = arxiv.Client(
        page_size=3,
        delay_seconds=3,
        num_retries=5,
    )

    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    papers = []

    try:
        for paper in client.results(search):
            papers.append(
                f"""
Title: {paper.title}

Authors: {", ".join(a.name for a in paper.authors)}

Published: {paper.published.date()}

Summary:
{paper.summary}
"""
            )

    except Exception as e:
        return f"Error: {e}"

    return  "\n".join(papers)

In [28]:
arxiv_tool.invoke({"query": "Transformers"})

'\nTitle: Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants\n\nAuthors: Jose I. Aizpurua\n\nPublished: 2025-12-20\n\nSummary:\nPower transformers are critical assets in power networks, whose reliability directly impacts grid resilience and stability. Traditional condition monitoring approaches, often rule-based or purely physics-based, struggle with uncertainty, limited data availability, and the complexity of modern operating conditions. Recent advances in machine learning (ML) provide powerful tools to complement and extend these methods, enabling more accurate diagnostics, prognostics, and control. In this two-part series, we examine the role of Neural Networks (NNs) and their extensions in transformer condition monitoring and health management tasks. This first paper introduces the basic concepts of NNs, explores Convolutional Neural Networks (CNNs) for condition monitoring using diverse data modalities, 

### Wikipedia Tool

In [20]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query:str):
    """This tool allows you to search for wikipedia for a query"""
    
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [25]:
wiki_tool.invoke("Anirudh Ravichander")

'Page: Anirudh Ravichander\nSummary: Anirudh Ravichander (born 16 October 1990), also credited mononymously as Anirudh, is an Indian composer and playback singer who works primarily in Tamil cinema, in addition to Telugu and Hindi films. He has won two Filmfare Awards South, ten SIIMA Awards, six Edison Awards and five Vijay awards.\nHis debut song "Why This Kolaveri Di", composed for the 2012 film 3, went viral across the globe and has achieved over 590 million views on YouTube. A.R. Murugadoss signed him to compose music for Kaththi (2014) starring Vijay & Samantha which included the viral hit "Selfie Pulla". The soundtrack for the film became Anirudh\'s highest profile soundtrack until he was signed to compose music for Rajinikanth\'s Petta in 2019.\nIn 2016, he signed a record deal with Sony Music, which publishes his independent albums and live concerts. In the same year, he featured with Diplo on the remix of Major Lazer\'s hit single "Cold Water".\n\nPage: Anirudh Ravichander di

### Custom Tool

In [21]:
@tool
def personal_info(name:str):
    """Use this tool to get personal information about someone"""

    info={
        "Balaji":"Balaji is an AI Engineer.",
        "Ganesh":"Ganesh is Data Scientist",
        "Rohit Sharma":"Rohit Sharma is a cricketer"
    }

    return info.get(name, "No information found for this person!!!")

In [23]:
personal_info.invoke("Rohit Sharma")

'Rohit Sharma is a cricketer'

### **Tool Binding**

In [29]:
tools = [duckduckgo_tool,arxiv_tool,wiki_tool,personal_info]

llm_with_tools = llm.bind_tools(tools)

In [32]:
response = llm_with_tools.invoke("I want papers related to transformers")
response.tool_calls

[{'name': 'arxiv_tool',
  'args': {'query': 'transformers'},
  'id': 'fc_76526174-7a73-4a16-a08f-6b8cd8c25825',
  'type': 'tool_call'}]